# Notebook 03 — SHAP Explainability
**AI Data Intelligence Copilot**

This notebook covers:
- SHAP TreeExplainer setup
- Global feature importance (summary plot)
- Individual prediction waterfall charts
- SHAP dependence plots
- Feature importance ranking DataFrame
- What-if simulation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.ingestion.schema_detector import detect_schema
from src.automl.preprocessor import preprocess
from src.automl.trainer import train_all
from src.explainability.shap_explainer import SHAPExplainer
from src.simulation.whatif_engine import simulate, plot_probability_gauge

print('Imports OK')

## 1. Train Best Model

In [ ]:
df = pd.read_csv('../data/sample_datasets/telco_churn.csv')
schema = detect_schema(df)
TARGET = 'Churn'

prep = preprocess(df, TARGET, schema)
results = train_all(
    prep.X_train, prep.X_test,
    prep.y_train, prep.y_test,
    schema.problem_type
)

best_model = results['best_model']['estimator']
print(f"Best model: {results['best_model']['model']}")

## 2. Compute SHAP Values

In [ ]:
explainer = SHAPExplainer(best_model, prep.X_train, schema.problem_type)
print('SHAP explainer ready')

## 3. Global Feature Importance

In [ ]:
importance_df = explainer.feature_importance_df()
print('Top 10 features by mean |SHAP|:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Plotly bar chart
explainer.feature_importance_plotly(top_n=15).show()

## 4. SHAP Summary Plot

In [ ]:
from IPython.display import Image
img_bytes = explainer.summary_plot_bytes(max_display=15)
Image(img_bytes)

## 5. Waterfall Chart — Individual Prediction

In [ ]:
# Show waterfall for row index 0
wf_bytes = explainer.waterfall_plot_bytes(instance_index=0)
Image(wf_bytes)

In [ ]:
# Try a few different rows
for idx in [5, 10, 25]:
    print(f'\n--- Row {idx} ---')
    actual = prep.y_test.iloc[idx]
    prob = best_model.predict_proba(prep.X_test.iloc[[idx]])[0][1]
    print(f'Actual label  : {actual}')
    print(f'Churn prob    : {prob:.4f}')
    Image(explainer.waterfall_plot_bytes(idx))

## 6. SHAP Dependence Plot

In [ ]:
top_feature = importance_df.iloc[0]['feature']
print(f'Dependence plot for top feature: {top_feature}')
explainer.dependence_plot_plotly(top_feature).show()

## 7. What-if Simulation

In [ ]:
# Pick a high-risk customer (high churn probability)
probas = best_model.predict_proba(prep.X_test)[:, 1]
high_risk_idx = probas.argsort()[-1]  # highest probability
base_row = prep.X_test.iloc[[high_risk_idx]]

print(f'High-risk row index: {high_risk_idx}')
print(f'Base churn probability: {probas[high_risk_idx]:.4f}')
print(base_row.T)

In [ ]:
# Simulate: what if we reduce the top numeric feature?
numeric_feats = [c for c in schema.numeric_cols if c in prep.X_test.columns]
top_num = numeric_feats[0] if numeric_feats else None

if top_num:
    current_val = float(base_row[top_num].values[0])
    new_val     = current_val * 0.7  # reduce by 30%

    result = simulate(
        best_model, base_row,
        overrides={top_num: new_val},
        problem_type=schema.problem_type
    )

    print(f'Feature changed  : {top_num}')
    print(f'Old value        : {current_val:.4f}')
    print(f'New value        : {new_val:.4f}')
    print(f'Original prob    : {result["original_value"]:.4f}')
    print(f'New prob         : {result["new_value"]:.4f}')
    print(f'Delta            : {result["delta"]:+.4f}')
    print(f'Recommendation   : {result["recommendation"]}')
    
    plot_probability_gauge(result['new_value']).show()

## 8. Batch Risk Scoring

In [ ]:
from src.recommendations.action_engine import batch_recommendations

rec_df = batch_recommendations(best_model, prep.X_test, top_n=10)
print('Top 10 highest-risk customers:')
print(rec_df.to_string(index=False))